In [1]:
# Librerias de HTTP para request y manipulación de datos
import requests
import pandas as pd 
pd.set_option('display.max_rows', None)
from IPython.display import HTML

# Conexión con MYSQL
import mysql.connector
from mysql.connector import Error

# Para limpiar los valores de API nulos
import numpy as np

# Variables de entorno (para contraseñas y datos sensibles)
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

# Manejo de archivos y datos JSON
import json

# Para quitar los mensajes de warnings
import warnings
warnings.filterwarnings("ignore")

# EXTRACCIÓN DE DATOS EN DEEZER Y LAST.FM

In [2]:
# Función genérica para extraer las API 
def extraccion_API(endpoint, verbose=True):

    try:
        # Endpoint como argumento de función
        datos = requests.get(endpoint)
        if datos.status_code == 200:
            # Se permite print detallado
            if verbose: 
                print ("API conectada correctamente")
            # Se convierte datos a JSON para que sea manejable por el resto de funciones
            datos_json = datos.json()
            return datos_json
        else:
            # En caso de error, se imprime fallo 
            print (f"Error de conexión en API {endpoint}: {datos.status_code}")

    # Error de conexión        
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # Error de tiempo de espera
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # Cubrimos el resto de errores
    except requests.exceptions.RequestException as e:
        print (e) 

In [3]:
# Lista los ID de los cantantes o grupos incluidos en el estudio
id_artistas = [12246, 160, 145, 564, 75491, 75798, 290, 483, 10803980, 1538640, 892, 10583405, 412, 13, 4050205, 384236, 119, 5620251, 259, 5962948, 196, 485, 425, 315929, 2446, 1755, 180, 98, 10977, 1434]

In [ ]:
# Se crea diccionario con artistas del estudio para crear grupos de control: femenino, masculino y mixto
diccionario_genero = {"Taylor Swift": "femenino",
                       "Shakira": "femenino", 
                       "Beyoncé": "femenino", 
                       "Rihanna": "femenino", 
                       "Lady Gaga": "femenino", 
                       "Adele": "femenino", 
                       "Madonna": "femenino", 
                       "Britney Spears": "femenino", 
                       "BLACKPINK": "femenino", 
                       "Little Mix":"femenino", 
                       "Coldplay": "masculino",     
                       "Bad Bunny": "masculino",     
                       "Queen": "masculino",     
                       "Eminem": "masculino",     
                       "The Weeknd": "masculino",     
                       "Ed Sheeran": "masculino",     
                       "Metallica": "masculino",     
                       "C. Tangana": "masculino",     
                       "Michael Jackson": "masculino",    
                        "Shawn Mendes": "masculino",     
                        "The Cranberries": "mixto",     
                        "Garbage": "mixto",     
                        "Blondie": "mixto",     
                        "Pretenders": "mixto",     
                        "Amaral": "mixto",     
                        "Roxette": "mixto",     
                        "ABBA": "mixto",     
                        "Evanescence": "mixto",    
                        "Paramore": "mixto", 
                        "La Oreja de Van Gogh": "mixto" }

In [4]:
# Función para extraer los datos de artistas de Deezer
def extraer_artistas(id_artistas):

    # Se define lista vacía y el contador de verbose
    lista_artistas = []
    contador = 0
    # Se recorre un endpoint por artista
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}"

        # Se lanza la función para extraer las API
        datos_artistas = extraccion_API(endpoint, verbose=False)
        contador += 1
        # Diccionario con los datos necesarios
        diccionario_artista = {
            "id_artista": datos_artistas["id"],
            "nombre": datos_artistas["name"]
            }
        lista_artistas.append(diccionario_artista)

    # Imprime el número de apis correctas
    print(f"{contador} APIs extraídas correctamente")
    # Unimos la lista de DF
    df_final = pd.DataFrame(lista_artistas)
    return df_final

In [5]:
# Recoge la variable de los artistas principales en el DF
df_artistas = extraer_artistas(id_artistas)
HTML(df_artistas.to_html(index=False))

30 APIs extraídas correctamente


id_artista,nombre
12246,Taylor Swift
160,Shakira
145,Beyoncé
564,Rihanna
75491,Lady Gaga
75798,Adele
290,Madonna
483,Britney Spears
10803980,BLACKPINK
1538640,Little Mix


In [6]:
# Funcion para comprobar que los artistas elegidos tienen 50 canciones en la API
def conteo_artistas(id_artistas):

    # Diccionario vacío para mostrar conteo
    diccionario_artistas = {}
    # Se recorren las APIs
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}/top?limit=50"
        datos = extraccion_API(endpoint, verbose=False)
        
        # Enfrenta artistas con el número de canciones en la API
        diccionario_artistas[artistas] = len(datos["data"])
    return diccionario_artistas

In [7]:
# Comprobación del número de canciones recogida en cada API
diccionario_artistas = conteo_artistas(id_artistas)
diccionario_artistas

{12246: 50,
 160: 50,
 145: 50,
 564: 50,
 75491: 50,
 75798: 50,
 290: 50,
 483: 50,
 10803980: 50,
 1538640: 50,
 892: 50,
 10583405: 50,
 412: 50,
 13: 50,
 4050205: 50,
 384236: 50,
 119: 50,
 5620251: 50,
 259: 50,
 5962948: 50,
 196: 50,
 485: 50,
 425: 50,
 315929: 50,
 2446: 50,
 1755: 50,
 180: 50,
 98: 50,
 10977: 50,
 1434: 50}

In [8]:
# Función para extraer los datos de las canciones de Deezer
def extraer_canciones(id_artistas):

    # Lista vacía para el DF y contador
    lista_canciones = []
    contador = 0
    # Se recorre un endpoint por artista
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}/top?limit=50"
        
        # Se lanza la función para extraer las API
        datos_canciones = extraccion_API(endpoint, verbose = False)
        contador += 1

        # Bucle para recoger los datos que necesitamos del JSON, incluido el booleano
        for cancion in datos_canciones["data"]:
            id_cancion = cancion["id"]
            id_artista = cancion["artist"]["id"]
            id_album = cancion["album"]["id"]
            titulo = cancion["title"]
            duracion = cancion["duration"]
            ranking = cancion["rank"]
            if len(cancion["contributors"]) == 1:
                colaboradores = False
            else:
                colaboradores = True

            # Diccionario creado con los datos extraidos del JSON
            diccionario_cancion = {
                    "id_cancion": id_cancion,
                    "id_artista": id_artista,
                    "id_album": id_album,
                    "title": titulo,
                    "duration": duracion,
                    "rank": ranking,
                    "contributors_bool": colaboradores
                }         
                        
            # Se añade el diccionario a la lista
            lista_canciones.append(diccionario_cancion)
    
    # Imprime el número de apis correctas
    print(f"{contador} APIs extraídas correctamente")        
    df_canciones = pd.DataFrame(lista_canciones)
    # Elimina posibles canciones duplicadas
    df_final = df_canciones.drop_duplicates(subset="id_cancion")
    return df_final, lista_canciones

In [9]:
# Extrae las variables de la función (DF y lista) y se recopilan los datos en un CSV
df_canciones ,  lista_canciones = extraer_canciones(id_artistas) 
df_canciones.to_csv("listado_de_canciones.csv", index=False)

30 APIs extraídas correctamente


In [ ]:
# Función para extraer los géneros de Deezer
def extraer_genero():

    # Endpoint único
    endpoint = "https://api.deezer.com/genre/"

    # Extrae los datos del endpoint. Verbose en True, solo es una API
    datos_genero = extraccion_API(endpoint, verbose = True)
    df_genero = pd.DataFrame(datos_genero["data"])
    # DF final solo con las columnas necesarias
    df_final = df_genero[["id","name"]]                       
    return df_final


In [11]:
# Se recoge el DF en la variable
df_genero = extraer_genero()
HTML(df_genero.to_html(index=False))

API conectada correctamente


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [12]:
# Función para extraer los álbumes de Deezer
def extraer_album():
    # Se crea conjunto para evitar duplicados
    album_ids_unicos = set()

    # Se recogen los albumes que aparecen en la lista de canciones ya extraida
    for cancion in lista_canciones:
        album_ids_unicos.add(cancion["id_album"])

    contador = 0
    lista_albumes = []
    # Extracción de datos de cada una de las API
    for album_id in album_ids_unicos:
        endpoint = f"https://api.deezer.com/album/{album_id}"
        datos_albumes = extraccion_API(endpoint, verbose = False)
        contador += 1
        
        diccionario_albumes = {
            "id_album": datos_albumes["id"],
            "id_artista": datos_albumes["artist"]["id"],
            "id_genre": datos_albumes["genre_id"],
            "titulo": datos_albumes["title"],
            "n_canciones": datos_albumes["nb_tracks"],
            "fecha_lanzamiento": datos_albumes["release_date"]
        }     

        # Añade los datos recogidos en el diccionario a la lista
        lista_albumes.append(diccionario_albumes)
    
    print(f"{contador} APIs extraídas correctamente") 
    df_final = pd.DataFrame(lista_albumes)
    return df_final
        


In [13]:
# Recoge el DF en la variable correspondiente
df_albumes = extraer_album()

591 APIs extraídas correctamente


In [ ]:
# Función para corregir los géneros musicales
def ajuste_genero():
    # Se verifican los géneros musicales faltantes
    generos_faltantes = set(df_albumes["id_genre"]) - set(df_genero["id"])
    lista_generos_nuevos = []
    # Extraemos la API de esos géneros
    for genero in generos_faltantes:
        endpoint = f"https://api.deezer.com/genre/{genero}"
        generos_extraidos = extraccion_API(endpoint, verbose=True)
        # Para API con error = género no existe. Se convierte el género en el "0" genérico
        if "error" in generos_extraidos:
            df_albumes.loc[df_albumes["id_genre"] == genero, "id_genre"] = 0
        else:    
            # Los géneros nuevos no incluidos en la lista de géneros original se extraen al DF
            lista_generos_nuevos.append(generos_extraidos)
    
    # Añade datos solo si hay valores que cambiar
    if len(lista_generos_nuevos) > 0:
        df_final = pd.DataFrame(lista_generos_nuevos)[["id", "name"]]
        return df_final


In [15]:
# Transforma df_albumes a CSV después de cambiar los erróneos.
df_albumes.to_csv("listado_de_albumes.csv", index=False)

# Extrae el DF del ajuste en una variable
df_ajuste_genero = ajuste_genero()

# Se concatenan a la tabla original
df_genero_final = pd.concat([df_genero, df_ajuste_genero], ignore_index=True)
HTML(df_genero_final.to_html(index=False))

API conectada correctamente
API conectada correctamente


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [16]:
# Función para extraer los artistas de LastFM
def extraer_last_fm(df_artistas):

    lista_artistas = []
    contador = 0
    # Ajusta los espacios a la forma de escritura de la API
    df_artistas["nombre"].str.replace(" ", "+")
    # Recorre las API con los artistas extraidos de Deezer
    for nombre in df_artistas["nombre"]:
        endpoint = f"https://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist={nombre}&api_key=68eddfdc6b072ad56527a4199d979038&format=json"
        datos_last_fm = extraccion_API(endpoint, verbose=False)
        contador += 1
    
        diccionario_artistas = {
            "nombre" : datos_last_fm["artist"]["name"],
            "oyentes" : datos_last_fm["artist"]["stats"]["listeners"],
            "reproducciones" : datos_last_fm["artist"]["stats"]["playcount"],
            "biografia" : datos_last_fm["artist"]["bio"]["summary"]
        }
            
        # Se añade artistas a la lista para extraer un nuevo DF
        lista_artistas.append(diccionario_artistas)

    df_final = pd.DataFrame(lista_artistas)
    # Corrige los saltos de párrafo en la biografía
    df_final["biografia"] = df_final["biografia"].str.replace("\n", " ")
    print(f"{contador} APIs extraídas correctamente") 
    return df_final
        


In [17]:
# Recoge en variable la extracción de LastFM de los artistas correspondientes al estudio
df_artistas_fm = extraer_last_fm(df_artistas)
df_artistas_fm.to_csv("listado_de_artistas_fm.csv", index=False)

30 APIs extraídas correctamente


In [18]:
# Localiza los id_artistas de colaboradores que aparezcan en canciones y álbumes
id_colaboradores = (set(df_albumes["id_artista"]) | set(df_canciones["id_artista"])) - set(id_artistas)
id_colaboradores

{11,
 230,
 359,
 413,
 556,
 1446,
 2896,
 3098,
 4088,
 4347,
 4474,
 4479,
 4962,
 5080,
 5828,
 7343,
 8631,
 9219,
 10226,
 12178,
 127322,
 165930,
 210977,
 246791,
 310260,
 380955,
 407188,
 554792,
 712271,
 1020109,
 1562681,
 1672366,
 3265001,
 3922661,
 3968561,
 4104927,
 4331004,
 4390053,
 4495513,
 4649104,
 4860761,
 4968870,
 5297021,
 5531258,
 5629748,
 5835993,
 5904266,
 6396188,
 6397900,
 7072729,
 7358224,
 7457468,
 7543848,
 7961888,
 8376040,
 8706544,
 9236850,
 9759672,
 9761322,
 9999412,
 11289472,
 12170972,
 12382106,
 12487862,
 67972932,
 79181242,
 108420982,
 207559207,
 213208547}

In [19]:
# Recoge en variable los datos de Deezer de los artistas colaboradores
df_colaboradores = extraer_artistas(id_colaboradores)

69 APIs extraídas correctamente


In [20]:
# Recoge en variable los datos de LastFM de los artistas colaboradores
df_colaboradores_fm = extraer_last_fm(df_colaboradores)

69 APIs extraídas correctamente


In [26]:
# Evita problemas de CaseSentitive 
df_artistas["nombre"] = df_artistas["nombre"].str.lower()
df_artistas_fm["nombre"] = df_artistas_fm["nombre"].str.lower()
df_colaboradores["nombre"] = df_colaboradores["nombre"].str.lower()
df_colaboradores_fm["nombre"] = df_colaboradores_fm["nombre"].str.lower()

# Combina los df de artistas principales de Deezer y LastFM
df_combinado_principales= pd.merge(df_artistas, df_artistas_fm, left_on="nombre", right_on="nombre")

# Combina los df de colaboradores principales de Deezer y LastFM
df_combinado_colaboradores = pd.merge(df_colaboradores, df_colaboradores_fm, left_on="nombre", right_on="nombre")

# Combina todos los artistas para la inserción en MYSQL
df_combinado_artistas = pd.concat([df_combinado_principales, df_combinado_colaboradores], ignore_index=True)

# Elimina duplicados
df_combinado_artistas = df_combinado_artistas.drop_duplicates(subset="id_artista")
df_combinado_artistas["nombre"] = df_combinado_artistas["nombre"].str.title()
df_combinado_artistas

,id_artista,nombre,oyentes,reproducciones,biografia
0,12246,Taylor Swift,5991121,3717857643,Taylor Alison Swift is an American singer-song...
1,160,Shakira,4941487,183501448,Shakira Isabel Mebarak Ripoll is a Colombian s...
2,145,Beyoncé,6435842,708291284,Beyoncé Giselle Knowles-Carter (born Septembe...
3,564,Rihanna,8360941,612399271,"Robyn Rihanna Fenty (born February 20, 1988), ..."
4,75491,Lady Gaga,7782251,1047571110,Stefani Joanne Angelina Germanotta (born 28 Ma...
5,75798,Adele,5666785,321532643,"Adele Laurie Blue Adkins MBE (born May 5, 1988..."
6,290,Madonna,5718934,372341424,"Madonna Louise Ciccone (born August 16, 1958) ..."
7,483,Britney Spears,6513441,501653702,"Britney Jean Spears (born December 2, 1981 in..."
8,10803980,Blackpink,2033081,318444975,BLACKPINK (Hangul: 블랙핑크; Katakana :ブラックピンク; st...
9,1538640,Little Mix,1872994,114925203,Little Mix is a British girl group formed in 2...


# CREACION DE BASE DE DATOS

In [74]:
# Función para conectar a MYSQL, recoge contraseña de variable de entorno
def conectar_mysql(host="127.0.0.1", user="root", password=password_sql, database=None):
    try:
        cnx = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database      
        )
        print("Conexión exitosa")
        # Devuelve la conexión para poder usarla posteriormente
        return cnx                      
    # Devuelve error de conexión en caso de haberlo
    except Error as e:
        print(f"Error al conectar: {e}")

In [76]:
# Recoge en variable la conexión de MYSQL
conexion = conectar_mysql()

Conexión exitosa


In [31]:
# Se define nombre para la base de datos
nombre_bd = "proyecto_music_stream_team1"

In [35]:
# Función para crear bases de datos general, nombre de base de datos como argumento
def crear_basededatos(nombre_bd):
   
    try:
        # Activa cursor solo mientras se usa la función
        with conexion.cursor() as cursor:
            query = f"CREATE DATABASE IF NOT EXISTS {nombre_bd}"
            # Ejecuta el cursor
            cursor.execute(query)
            print ("Query exitosa")
 
    # Imprime error en caso de haberlo
    except Error as e:
        print (f"Error creando base de datos: {e}")

In [36]:
# Se lanza base de datos con el nombre como argumento
crear_basededatos(nombre_bd)

Query exitosa


In [29]:
# Función para borrar la base de datos si es necesario
def borrar_base_datos(db_name):

    try:
        # Solicita confirmación a usuario
        respuesta_usuario = input (f"La base de datos '{db_name}' será borrada, ¿Estás seguro? (Y/N):").upper()
        # Actúa solo en caso positivo
        if respuesta_usuario == "Y":
            with conexion.cursor() as cursor:
                cursor.execute(f"DROP DATABASE IF EXISTS {db_name}")
                conexion.commit()
                print(f"Base de datos {db_name} eliminada correctamente")
        else:
            # Cualquier otra opción cancela el borrado
            print ("Operación cancelada")
    
    # Captura posibles errores en MYSQL
    except Error as e:
        print(f"Error al eliminar la base de datos: {e}")

In [40]:
# Lanza la función para borrar
borrar_base_datos(nombre_bd)

Operación cancelada


In [41]:
# Función para crear tablas general
def crear_tablas_genericas(nombre_bd, nombre_tabla, tabla_esquema):
   
    try:
        # Activa cursor
        with conexion.cursor() as cursor:
            cursor.execute(f"USE {nombre_bd};")
            # Se define la query general con nombre de tabla y esquema como argumentos
            query = f''' CREATE TABLE IF NOT EXISTS {nombre_tabla} ({tabla_esquema});'''
            # Ejecuta la petición
            cursor.execute(query)
            print ("Query creación exitosa")
   
    # Recoge los errores posibles en MYSQL
    except Error as e:
        print (f"Error creando tabla: {e}")

In [42]:
# Se definen las tablas necesarias como variables
tabla_artista = 'artista'
tabla_genero_musical = 'genero_musical'
tabla_canciones = 'canciones'
tabla_album = 'album'

In [43]:
# Esquema de la tabla artistas
esquema_artista = '''id_artista INT PRIMARY KEY,
   nombre VARCHAR(30) NOT NULL,
   oyentes INT,
   reproducciones BIGINT,
   biografia VARCHAR(1000) NOT NULL,
   genero VARCHAR(10),
   artista_principal BOOLEAN
   '''

In [44]:
# Esquema de la tabla género musical
esquema_genero_musical = '''id_genero INT PRIMARY KEY, 
nombre VARCHAR(40)'''

In [45]:
# Esquema de la tabla canciones
esquema_canciones = '''id_cancion BIGINT PRIMARY KEY,
id_artista INT NOT NULL,
id_album BIGINT NOT NULL,
titulo VARCHAR(200) NOT NULL,
duracion INT,
ranking_lista INT,
colaboraciones BOOLEAN,
FOREIGN KEY (id_artista)
   REFERENCES artista(id_artista),
FOREIGN KEY (id_album)
   REFERENCES album(id_album)
'''

In [46]:
# Esquema de la tabla album
esquema_album = ''' id_album BIGINT PRIMARY KEY,
id_artista INT,
id_genero INT,
titulo VARCHAR(200) NOT NULL,
numero_canciones INT,
fecha_lanzamiento DATE,
FOREIGN KEY (id_genero)
   REFERENCES genero_musical(id_genero),
FOREIGN KEY (id_artista)
   REFERENCES artista(id_artista)
'''

In [47]:
# Creación de la tabla genero 
crear_tablas_genericas(nombre_bd, tabla_genero_musical, esquema_genero_musical)

Query creación exitosa


In [48]:
# Creación de la tabla artista 
crear_tablas_genericas(nombre_bd,tabla_artista, esquema_artista)

Query creación exitosa


In [49]:
# Creación de la tabla album 
crear_tablas_genericas(nombre_bd,tabla_album, esquema_album)

Query creación exitosa


In [50]:
# Creación de la tabla canciones 
crear_tablas_genericas(nombre_bd,tabla_canciones, esquema_canciones)

Query creación exitosa


# INSERCIÓN EN MYSQL

In [51]:
# Abrimos cursor para la inserción
cursor = conexion.cursor()

In [52]:
# Se insertan datos del DF de artistas combinado a MYSQL
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_artista} (id_artista, nombre, oyentes, reproducciones, biografia)
VALUES (%s, %s, %s, %s, %s)'''
df_limpio = df_combinado_artistas.replace({np.nan: None, 'nan': None, 'Nan': None}) #para quitar informacion vacia
df_limpio = df_limpio[['id_artista','nombre','oyentes','reproducciones','biografia']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [53]:
# Se insertan datos de todos los géneros musicales a MYSQL
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_genero_musical} (id_genero, nombre)
VALUES (%s, %s)'''
df_limpio = df_genero_final.replace({np.nan: None, 'nan': None, 'Nan': None}) 
df_limpio = df_limpio[['id','name']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [54]:
# Se insertan datos de albumes a MYSQL
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_album} (id_album, id_artista, id_genero, titulo, numero_canciones, fecha_lanzamiento)
VALUES (%s, %s, %s, %s, %s, %s)'''
df_limpio = df_albumes.replace({np.nan: None, 'nan': None, 'Nan': None}) 
df_limpio = df_limpio[['id_album', 'id_artista', 'id_genre', 'titulo', 'n_canciones', 'fecha_lanzamiento']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [55]:
# Se insertan datos de canciones a MYSQL
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_canciones} (id_cancion, id_artista, id_album, titulo, duracion, ranking_lista, colaboraciones)
VALUES (%s, %s, %s, %s, %s, %s, %s)'''

df_limpio = df_canciones.replace({np.nan: None, 'nan': None, 'Nan': None}) #para quitar informacion vacia
df_limpio = df_limpio[['id_cancion', 'id_artista', 'id_album', 'title', 'duration', 'rank', 'contributors_bool']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [57]:
# Se ejecuta inserción de datos en NMYSQL de género para control
cursor.execute(f'''USE {nombre_bd};''')
for nombre, genero in diccionario_genero.items(): 
    cursor.execute(f'''UPDATE {tabla_artista} SET genero = %s WHERE nombre = %s''', (genero, nombre))
conexion.commit()

In [58]:
# Se ejecuta inserción de datos de la tabla "artista_principal" para aquellos que pertenecen al estudio
cursor.execute(f'''UPDATE {tabla_artista} SET artista_principal = TRUE WHERE id_artista IN ({','.join(['%s']*len(id_artistas))})''', id_artistas)
cursor.execute(f'''UPDATE {tabla_artista} SET artista_principal = FALSE WHERE id_artista NOT IN ({','.join(['%s']*len(id_artistas))})''', id_artistas)
conexion.commit()

In [59]:
# Cierra el cursor activo
cursor.close()

True

# ANALISIS Y CONSULTAS DE BASE DE DATOS

¿Suena igual para todos? Análisis de la presencia y popularidad femenina en las plataformas de streaming musical
¿Las artistas femeninas tienen la misma representación y éxito que los artistas masculinos en la música digital?

1. ¿Cuál es la media de oyentes y reproducciones de las artistas femeninas frente a los masculinos?
2. Eficiencia por canción: Si dividimos reproducciones entre el número total de canciones, ¿quién obtiene más "rendimiento" por cada tema lanzado?
3. El Techo del Ranking: ¿Cuál es la posición media en ranking_lista para las mujeres frente a los hombres?
4. Presencia en el Top: ¿Qué porcentaje de artistas en el "Top 10" (según ranking_lista) son mujeres?
5. El fenómeno de la colaboración: Según la columna colaboraciones de la tabla canciones, ¿quién colabora más? ¿Las mujeres suelen aparecer más en canciones con colaboraciones que en solitario?
6. Densidad de los álbumes: ¿Quién saca álbumes más largos (más número canciones)?
7. Frecuencia de lanzamiento: Usando fecha_lanzamiento, ¿cuál es el tiempo medio que pasa una mujer entre álbum y álbum frente a los hombres?
8. Géneros "Generizados": ¿En qué estilos musicales (nombre de la tabla genero_musical) hay una presencia nula o mínima de mujeres?


In [ ]:
# Función para ejecutar las diferentes consultas
def consultas_generales(nombre_bd, query_sql):
   
    try:
        # Abre el cursor para realizar las con sultas
        with conexion.cursor() as cursor:
            # Selecciona la base de datos
            cursor.execute(f"USE {nombre_bd};")
            # Ejecuta la query y devuelve la consulta
            df_query = pd.read_sql(query_sql, conexion)
            return df_query
   
    # Captura errores en MySQL
    except Error as e:
        print (f"Error en la consulta: {e}")

In [ ]:
# 1. ¿Cuál es la media de oyentes y reproducciones de las artistas femeninas frente a los masculinos?
query_1 = '''SELECT genero, 
    ROUND(AVG(reproducciones), 2) AS media_reproducciones,    
    ROUND(AVG(oyentes), 2) AS media_oyentes
FROM artista
WHERE artista_principal = TRUE
GROUP BY genero
ORDER BY media_reproducciones DESC;'''

df_query = consultas_generales(nombre_bd, query_1)
HTML(df_query.to_html(index=False))


genero,media_reproducciones,media_oyentes
femenino,789851870.3,5531687.7
masculino,471223630.6,5206266.0
mixto,140344746.4,2915494.1


In [ ]:
# 1.1 ¿Quién obtiene más "rendimiento" por cada tema lanzado?
query_1_1 = '''SELECT ROW_NUMBER() OVER (ORDER BY AVG(c.ranking_lista) DESC) AS posicion, 
    a.nombre, 
    a.genero,
    AVG(c.ranking_lista) AS promedio_ranking 
FROM canciones c 
JOIN artista a ON c.id_artista = a.id_artista 
WHERE a.artista_principal = TRUE 
GROUP BY a.nombre, a.genero
ORDER BY promedio_ranking DESC 
LIMIT 30;'''
df_query = consultas_generales(nombre_bd, query_1_1)
HTML(df_query.to_html(index=False))

# Hemos seleccionado el promedio de ranking donde se ordena los artistas principales con mayor ranking

posicion,nombre,genero,promedio_ranking
1,Michael Jackson,masculino,856553.4694
2,Bad Bunny,masculino,819557.9333
3,The Weeknd,masculino,768402.5610
4,Rihanna,femenino,765139.8723
5,Taylor Swift,femenino,751834.2857
6,Lady Gaga,femenino,733596.7872
7,Ed Sheeran,masculino,719023.8478
8,Eminem,masculino,709177.4792
9,Shakira,femenino,700886.4324
10,Coldplay,masculino,694455.4490


In [ ]:
# 2. ¿Cuál es la posición media en ranking_lista para las mujeres frente a los hombres?
query_2 = '''SELECT a.genero,
    ROUND(AVG(c.ranking_lista), 2) AS posicion_media
FROM artista a
JOIN canciones c ON a.id_artista = c.id_artista
WHERE c.ranking_lista IS NOT NULL AND a.artista_principal = TRUE 
GROUP BY a.genero
ORDER BY posicion_media DESC;'''
df_query = consultas_generales(nombre_bd, query_2)
HTML(df_query.to_html(index=False))

genero,posicion_media
masculino,675245.12
femenino,645687.66
mixto,395509.79


In [64]:
# 2.1 ¿Qué porcentaje de artistas en el "Top 10" (según ranking_lista) son mujeres?
query_2_1 = '''SELECT ROUND(100 * COUNT(DISTINCT CASE WHEN a.genero = 'femenino' THEN a.id_artista END) / COUNT(DISTINCT a.id_artista), 2) AS porcentaje_mujeres_top10
FROM artista a
JOIN (
    SELECT id_artista, AVG(ranking_lista) AS promedio_ranking
    FROM canciones
    WHERE id_artista IN (SELECT id_artista FROM artista WHERE artista_principal = TRUE)
    GROUP BY id_artista
    ORDER BY promedio_ranking DESC
    LIMIT 10
) top10 ON a.id_artista = top10.id_artista;'''
df_query = consultas_generales(nombre_bd, query_2_1)
HTML(df_query.to_html(index=False))

porcentaje_mujeres_top10
40.0


In [65]:
# 2.2 ¿Qué porcentaje de artistas en el "Top 10" (según ranking_lista) son grupos mixtos con mujeres al frente?
query_2_2 = '''SELECT ROUND(100 * COUNT(DISTINCT CASE WHEN a.genero = 'mixto' THEN a.id_artista END) / COUNT(DISTINCT a.id_artista), 2) AS porcentaje_mujeres_top10
FROM artista a
JOIN (
    SELECT id_artista, AVG(ranking_lista) AS promedio_ranking
    FROM canciones
    WHERE id_artista IN (SELECT id_artista FROM artista WHERE artista_principal = TRUE)
    GROUP BY id_artista
    ORDER BY promedio_ranking DESC
    LIMIT 10
) top10 ON a.id_artista = top10.id_artista;'''
df_query = consultas_generales(nombre_bd, query_2_2)
HTML(df_query.to_html(index=False))

porcentaje_mujeres_top10
0.0


In [ ]:
# 3. ¿Quién colabora más? ¿Las mujeres suelen aparecer más en canciones con colaboraciones que en solitario?

query_3_1 = '''SELECT a.nombre, a.genero,
    COUNT(*) AS canciones_con_colaboracion 
FROM canciones c 
JOIN artista a ON c.id_artista = a.id_artista
WHERE c.colaboraciones = TRUE AND a.artista_principal = TRUE
GROUP BY a.nombre, a.genero
ORDER BY canciones_con_colaboracion DESC
LIMIT 30;'''
df_query = consultas_generales(nombre_bd, query_3_1)
HTML(df_query.to_html(index=False))

nombre,genero,canciones_con_colaboracion
C. Tangana,masculino,27
Eminem,masculino,19
Shakira,femenino,19
Bad Bunny,masculino,16
The Weeknd,masculino,14
Rihanna,femenino,13
Beyoncé,femenino,12
Lady Gaga,femenino,12
The Cranberries,mixto,10
Ed Sheeran,masculino,10


In [ ]:
# 3.2 Comparativa por grupo
query_3_2 = '''SELECT a.genero,
    COUNT(*) AS canciones_con_colaboracion 
FROM canciones c 
JOIN artista a ON c.id_artista = a.id_artista
WHERE c.colaboraciones = TRUE AND a.artista_principal = TRUE
GROUP BY a.genero
ORDER BY canciones_con_colaboracion DESC;'''
df_query = consultas_generales(nombre_bd, query_3_2)
HTML(df_query.to_html(index=False))

genero,canciones_con_colaboracion
masculino,110
femenino,86
mixto,22


In [ ]:
# 4.1. ¿Quién saca álbumes más largos (más numero_canciones)?
    # Se filtran singles y albumes tipo edición limitada con el filtro de cantidad
query_4_1 ='''SELECT a.nombre, a. genero, 
    AVG(al.numero_canciones) AS media_canciones_por_album
FROM album al
JOIN artista a ON al.id_artista = a.id_artista
WHERE a.artista_principal = TRUE AND al.numero_canciones >1 AND al.numero_canciones <=30
GROUP BY a.nombre, a. genero
ORDER BY media_canciones_por_album DESC;'''
df_query = consultas_generales(nombre_bd, query_4_1)
HTML(df_query.to_html(index=False))

nombre,genero,media_canciones_por_album
Eminem,masculino,19.8667
Bad Bunny,masculino,19.6000
Little Mix,femenino,17.7500
Amaral,mixto,17.7273
Michael Jackson,masculino,17.6667
Beyoncé,femenino,16.6429
Taylor Swift,femenino,16.5000
Britney Spears,femenino,16.3333
The Cranberries,mixto,15.3333
Blondie,mixto,15.2941


In [79]:
# 4.1 ¿Cuál es el tiempo medio que pasa una mujer entre álbum y álbum frente a los hombres?
query_4_2 = '''SELECT
    a.genero,
    AVG(DATEDIFF(al.fecha_lanzamiento, prev.fecha_lanzamiento)) AS media_dias_entre_albumes
FROM album al
JOIN artista a 
    ON al.id_artista = a.id_artista
JOIN album prev 
    ON al.id_artista = prev.id_artista
    AND al.fecha_lanzamiento > prev.fecha_lanzamiento
WHERE a.artista_principal = TRUE
AND NOT EXISTS (
    SELECT 1 
    FROM album medio
    WHERE medio.id_artista = al.id_artista
      AND medio.fecha_lanzamiento > prev.fecha_lanzamiento
      AND medio.fecha_lanzamiento < al.fecha_lanzamiento
)
GROUP BY a.genero;'''
df_query = consultas_generales(nombre_bd, query_4_2)
HTML(df_query.to_html(index=False))


genero,media_dias_entre_albumes
masculino,415.3619
mixto,771.5750
femenino,487.8228


In [70]:
# 5. ¿En qué estilos musicales (nombre de la tabla genero_musical) hay una presencia nula o mínima de mujeres?
query_5 = '''SELECT g.nombre AS genero_musical,
    COUNT(DISTINCT a.id_artista) AS total_artistas,
    ROUND(100 * COUNT(DISTINCT CASE WHEN a.genero = 'femenino' THEN a.id_artista END) / COUNT(DISTINCT a.id_artista), 2) AS porcentaje_femenino,
    ROUND(100 * COUNT(DISTINCT CASE WHEN a.genero = 'masculino' THEN a.id_artista END) / COUNT(DISTINCT a.id_artista), 2) AS porcentaje_masculino,
    ROUND(100 * COUNT(DISTINCT CASE WHEN a.genero = 'mixto' THEN a.id_artista END) / COUNT(DISTINCT a.id_artista), 2) AS porcentaje_mixto
FROM genero_musical g
JOIN album al ON g.id_genero = al.id_genero
JOIN artista a ON al.id_artista = a.id_artista
WHERE a.artista_principal = TRUE
GROUP BY g.nombre
ORDER BY porcentaje_femenino ASC;'''
df_query = consultas_generales(nombre_bd, query_5)
HTML(df_query.to_html(index=False))

genero_musical,total_artistas,porcentaje_femenino,porcentaje_masculino,porcentaje_mixto
Metal,1,0.00,100.00,0.00
Pop latino,2,0.00,0.00,100.00
Reggaeton,1,0.00,100.00,0.00
Rock,10,10.00,30.00,60.00
Alternativo,10,30.00,10.00,60.00
Películas/Juegos,6,33.33,50.00,16.67
Pop,22,45.45,22.73,31.82
Dance,6,50.00,50.00,0.00
Latino,2,50.00,50.00,0.00
Rap/Hip Hop,4,50.00,50.00,0.00


In [71]:
# Función para cerrar la conexión con MYSQL
def cerrar_mysql():

    try:
        conexion.close()
        print("Desconectado correctamente")
    
    # Devuelve errores de MYSQL que impidan el cierre
    except Error as e:
        print (f"Ha habido un error: {e}")

In [80]:
cerrar_mysql()

Desconectado correctamente
